In [1]:
# import modules
import numpy as np
import pandas as pd
from typing import Set, List, Dict

# make sure its using colab to connect with google drive for storing output?
import os
import re
import ast
from google.colab import drive

In [2]:
# preparing the orginal leptin fragments sequence
LeptinEff = "IQKVQDDTKTLIKTIVTRINDISHTQSVSSKQKVTGLDFIPGLHPILTLSKMDQTLAVYQQILTSMPSRNVIQISNDLENLRDLLHVLAFSKSCHLPWASGLETLDSLGGVLEASGYSTEVVALSRLQGSLQDMLWQLDLSPGC"
OGLeptin_Frag = "SCHLPWASGLETLDS"

# hexapeptide
Leptin116121 = "SCHLPW"
Leptin117122 = "CHLPWA"
Leptin118123 = "HLPWAS"
Leptin119124 = "LPWASG"
Leptin120125 = "PWASGL"
Leptin121126 = "WASGLE"
Leptin122127 = "ASGLET"
Leptin123128 = "SGLETL"
Leptin124129 = "GLETLD"
Leptin125130 = "LETLDS"

# Nonapeptide
Leptin116124 = "SCHLPWASG"
Leptin117125 = "CHLPWASGL"
Leptin118126 = "HLPWASGLE"
Leptin119127 = "LPWASGLET" 
Leptin120128 = "PWASGLETL"
Leptin121129 = "WASGLETLD"
Leptin122130 = "ASGLETLDS"

In [3]:
# modified leptin dictionary
fragment_map = {
    'C7': (117, 122),  #Leptin117122 
    'S6': (116, 121),  #Leptin116121
    'H8': (118, 123),  #Leptin118123 
    'L9': (119, 124),  #Leptin119124
    'P1': (120, 125)   #Leptin120125
}

In [4]:
# Amino acid identity dictionary
amino_acids = {
    'A': {'name': 'Alanine', 'codons': ['GCU','GCC','GCA','GCG'], 'mw': 89.09},
    'R': {'name': 'Arginine', 'codons': ['CGU','CGC','CGA','CGG','AGA','AGG'], 'mw': 174.20},
    'N': {'name': 'Asparagine', 'codons': ['AAU','AAC'], 'mw': 132.12},
    'D': {'name': 'Aspartic acid', 'codons': ['GAU','GAC'], 'mw': 133.10},
    'C': {'name': 'Cysteine', 'codons': ['UGU','UGC'], 'mw': 121.16},
    'Q': {'name': 'Glutamine', 'codons': ['CAA','CAG'], 'mw': 146.15},
    'E': {'name': 'Glutamic acid', 'codons': ['GAA','GAG'], 'mw': 147.13},
    'G': {'name': 'Glycine', 'codons': ['GGU','GGC','GGA','GGG'], 'mw': 75.07},
    'H': {'name': 'Histidine', 'codons': ['CAU','CAC'], 'mw': 155.16},
    'I': {'name': 'Isoleucine', 'codons': ['AUU','AUC','AUA'], 'mw': 131.17},
    'L': {'name': 'Leucine', 'codons': ['UUA','UUG','CUU','CUC','CUA','CUG'], 'mw': 131.17},
    'K': {'name': 'Lysine', 'codons': ['AAA','AAG'], 'mw': 146.19},
    'M': {'name': 'Methionine', 'codons': ['AUG'], 'mw': 149.21},
    'F': {'name': 'Phenylalanine', 'codons': ['UUU','UUC'], 'mw': 165.19},
    'P': {'name': 'Proline', 'codons': ['CCU','CCC','CCA','CCG'], 'mw': 115.13},
    'S': {'name': 'Serine', 'codons': ['UCU','UCC','UCA','UCG','AGU','AGC'], 'mw': 105.09},
    'T': {'name': 'Threonine', 'codons': ['ACU','ACC','ACA','ACG'], 'mw': 119.12},
    'W': {'name': 'Tryptophan', 'codons': ['UGG'], 'mw': 204.23},
    'Y': {'name': 'Tyrosine', 'codons': ['UAU','UAC'], 'mw': 181.19},
    'V': {'name': 'Valine', 'codons': ['GUU','GUC','GUA','GUG'], 'mw': 117.15}
}

In [19]:
# data of the filtered assessment - original fragments

# OG
filtered_avg_001146 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_whole.csv?raw=true'
filtered_avg_116130 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_116130.csv?raw=true'
filtered_med_001146 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_whole.csv?raw=true'
filtered_med_116130 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_116130.csv?raw=true'

# hexapeptide - avg data
filtered_avg_116121 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_116121.csv?raw=true'
filtered_avg_117122 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_117122.csv?raw=true'
filtered_avg_118123 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_118123.csv?raw=true'
filtered_avg_119124 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_119124.csv?raw=true'
filtered_avg_120125 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_120125.csv?raw=true'
filtered_avg_121126 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_121126.csv?raw=true'
filtered_avg_122127 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_122127.csv?raw=true'
filtered_avg_123128 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_123128.csv?raw=true'
filtered_avg_124129 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_124129.csv'
filtered_avg_125130 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_125130.csv?raw=true'


# hexapeptide - med data
filtered_med_116121 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_116121.csv?raw=true'
filtered_med_117122 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_117122.csv?raw=true'
filtered_med_118123 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_118123.csv?raw=true'
filtered_med_119124 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_119124.csv?raw=true'
filtered_med_120125 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_120125.csv?raw=true'
filtered_med_121126 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_121126.csv?raw=true'
filtered_med_122127 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_122127.csv?raw=true'
filtered_med_123128 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_123128.csv?raw=true'
filtered_med_124129 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_124129.csv?raw=true'
filtered_med_125130 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_125130.csv?raw=true'

# nonapeptide - avg data
filtered_avg_116124 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_116124.csv?raw=true'
filtered_avg_117125 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_117125.csv?raw=true'
filtered_avg_118126 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_118126.csv?raw=true'
filtered_avg_119127 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_119127.csv?raw=true'
filtered_avg_120128 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_120128.csv?raw=true'
filtered_avg_121129 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_121129.csv?raw=true'
filtered_avg_122130 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_avg_122130.csv?raw=true'

# nonapeptide - med data
filtered_med_116124 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_116124.csv?raw=true'
filtered_med_117125 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_117125.csv?raw=true'
filtered_med_118126 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_118126.csv?raw=true'
filtered_med_119127 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_119127.csv?raw=true'
filtered_med_120128 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_120128.csv?raw=true'
filtered_med_121129 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_121129.csv?raw=true'
filtered_med_122130 = 'https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_filtered/filtered_med_122130.csv?raw=true'

In [6]:
# data of the filtered assessment - modified 116 - 121

# med
filtered_med_SS6V = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6V.csv?raw=true"
filtered_med_SS6L = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6L.csv?raw=true"
filtered_med_SS6I = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6I.csv?raw=true"
filtered_med_SS6F = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6F.csv?raw=true"
filtered_med_SS6M = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6M.csv?raw=true"
filtered_med_SS6W = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6W.csv?raw=true"
filtered_med_SS6P = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6P.csv?raw=true"
filtered_med_SS6G = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6G.csv?raw=true"
filtered_med_SS6C = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6C.csv?raw=true"
filtered_med_SS6R = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6R.csv?raw=true"
filtered_med_SS6N = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6N.csv?raw=true"
filtered_med_SS6D = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6D.csv?raw=true"
filtered_med_SS6E = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6E.csv?raw=true"
filtered_med_SS6Q = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6Q.csv?raw=true"
filtered_med_SS6H = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6H.csv?raw=true"
filtered_med_SS6K = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6K.csv?raw=true"
filtered_med_SS6A = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6A.csv?raw=true"
filtered_med_SS6T = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6T.csv?raw=true"
filtered_med_SS6Y = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_SS6Y.csv?raw=true"

# avg
filtered_avg_SS6V = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6V.csv?raw=true"
filtered_avg_SS6L = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6L.csv?raw=true"
filtered_avg_SS6I = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6I.csv?raw=true"
filtered_avg_SS6F = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6F.csv?raw=true"
filtered_avg_SS6M = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6M.csv?raw=true"
filtered_avg_SS6W = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6W.csv?raw=true"
filtered_avg_SS6P = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6P.csv?raw=true"
filtered_avg_SS6G = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6G.csv?raw=true"
filtered_avg_SS6C = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6C.csv?raw=true"
filtered_avg_SS6R = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6R.csv?raw=true"
filtered_avg_SS6N = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6N.csv?raw=true"
filtered_avg_SS6D = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6D.csv?raw=true"
filtered_avg_SS6E = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6E.csv?raw=true"
filtered_avg_SS6Q = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6Q.csv?raw=true"
filtered_avg_SS6H = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6H.csv?raw=true"
filtered_avg_SS6K = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6K.csv?raw=true"
filtered_avg_SS6A = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6A.csv?raw=true"
filtered_avg_SS6T = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6T.csv?raw=true"
filtered_avg_SS6Y = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_SS6Y.csv?raw=true"


In [7]:
# data of the filtered assessment - modified 117 - 122

# avg data
filtered_avg_AC7V = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7V.csv?raw=true"
filtered_avg_AC7L = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7L.csv?raw=true"
filtered_avg_AC7I = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7I.csv?raw=true"
filtered_avg_AC7F = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7F.csv?raw=true"
filtered_avg_AC7M = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7M.csv?raw=true"
filtered_avg_AC7W = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7W.csv?raw=true"
filtered_avg_AC7P = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7P.csv?raw=true"
filtered_avg_AC7G = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7G.csv?raw=true"
filtered_avg_AC7C = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7C.csv?raw=true"
filtered_avg_AC7R = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7R.csv?raw=true"
filtered_avg_AC7N = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7N.csv?raw=true"
filtered_avg_AC7D = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7D.csv?raw=true"
filtered_avg_AC7E = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7E.csv?raw=true"
filtered_avg_AC7Q = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7Q.csv?raw=true"
filtered_avg_AC7H = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7H.csv?raw=true"
filtered_avg_AC7K = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7K.csv?raw=true"
filtered_avg_AC7S = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7S.csv?raw=true"
filtered_avg_AC7T = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7T.csv?raw=true"
filtered_avg_AC7Y = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_avg_AC7Y.csv?raw=true"

# med data
filtered_med_AC7V = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7V.csv?raw=true"
filtered_med_AC7L = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7L.csv?raw=true"
filtered_med_AC7I = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7I.csv?raw=true"
filtered_med_AC7F = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7F.csv?raw=true"
filtered_med_AC7M = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7M.csv?raw=true"
filtered_med_AC7W = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7W.csv?raw=true"
filtered_med_AC7P = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7P.csv?raw=true"
filtered_med_AC7G = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7G.csv?raw=true"
filtered_med_AC7C = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7C.csv?raw=true"
filtered_med_AC7R = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7R.csv?raw=true"
filtered_med_AC7N = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7N.csv?raw=true"
filtered_med_AC7D = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7D.csv?raw=true"
filtered_med_AC7E = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7E.csv?raw=true"
filtered_med_AC7Q = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7Q.csv?raw=true"
filtered_med_AC7H = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7H.csv?raw=true"
filtered_med_AC7K = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7K.csv?raw=true"
filtered_med_AC7S = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7S.csv?raw=true"
filtered_med_AC7T = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7T.csv?raw=true"
filtered_med_AC7Y = "https://github.com/FairzanIsLearning/OralLeptinExploration/blob/main/Ligand%20assessment/PRM_score_modified_filtered/filtered_med_AC7Y.csv?raw=true"

In [8]:
# data of the filtered assessment - modified 118 - 123

# med
filtered_med_AH8V = "?raw=true"
filtered_med_AH8L = "?raw=true"
filtered_med_AH8I = "?raw=true"
filtered_med_AH8F = "?raw=true"
filtered_med_AH8M = "?raw=true"
filtered_med_AH8W = "?raw=true"
filtered_med_AH8P = "?raw=true"
filtered_med_AH8G = "?raw=true"
filtered_med_AH8C = "?raw=true"
filtered_med_AH8R = "?raw=true"
filtered_med_AH8N = "?raw=true"
filtered_med_AH8D = "?raw=true"
filtered_med_AH8E = "?raw=true"
filtered_med_AH8Q = "?raw=true"
filtered_med_AH8H = "?raw=true"
filtered_med_AH8K = "?raw=true"
filtered_med_AH8S = "?raw=true"
filtered_med_AH8T = "?raw=true"
filtered_med_AH8Y = "?raw=true"

# Avg
filtered_avg_AH8V = "?raw=true"
filtered_avg_AH8L = "?raw=true"
filtered_avg_AH8I = "?raw=true"
filtered_avg_AH8F = "?raw=true"
filtered_avg_AH8M = "?raw=true"
filtered_avg_AH8W = "?raw=true"
filtered_avg_AH8P = "?raw=true"
filtered_avg_AH8G = "?raw=true"
filtered_avg_AH8C = "?raw=true"
filtered_avg_AH8R = "?raw=true"
filtered_avg_AH8N = "?raw=true"
filtered_avg_AH8D = "?raw=true"
filtered_avg_AH8E = "?raw=true"
filtered_avg_AH8Q = "?raw=true"
filtered_avg_AH8H = "?raw=true"
filtered_avg_AH8K = "?raw=true"
filtered_avg_AH8S = "?raw=true"
filtered_avg_AH8T = "?raw=true"
filtered_avg_AH8Y = "?raw=true"

In [9]:
# save the output to google folder
# Mount Google Drive (only once per session)
drive.mount('/content/drive')

# Define the target folder 
results_folder = '/content/drive/MyDrive/Master/Barcelona/Thesis/Documentation/Fit_comparison_native'  
os.makedirs(results_folder, exist_ok=True)

folder_link = 'https://drive.google.com/drive/u/2/folders/1HBP1qiokpx1KiOKbGwLzTgS9sMcyugG-'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# function to extract the fragment sequence

# for native hexapeptide
def extract_fragment_id(filename: str) -> str:
    """
    Extract fragment identifier from filename.
    Expects a 6-digit number where first3=start, last3=end.
    Example: 'LepR_602608.csv' -> '602-608'
    """
    match = re.search(r'(\d{6})', filename)
    if not match:
        raise ValueError(f"Filename {filename} does not contain a 6-digit number")
    digits = match.group(1)
    start, end = digits[:3], digits[3:]
    return f"{start}-{end}"

# for modified hexapeptide
def extract_fragment_mod_id(code):
    """
    Parse a 4‑character modification code (e.g., 'AC7V').
    Returns a dictionary with all details.
    """
    if len(code) != 4:
        raise ValueError(f"Code must be exactly 4 characters, got '{code}'")
    orig = code[0]
    frag_code = code[1:3]
    subs = code[3]
    if orig not in amino_acids:
        raise ValueError(f"Unknown original amino acid: '{orig}'")
    if subs not in amino_acids:
        raise ValueError(f"Unknown substituted amino acid: '{subs}'")
    if frag_code not in fragment_map:
        raise ValueError(f"Unknown fragment code: '{frag_code}'")
    start, end = fragment_map[frag_code]
    return {
        'original_aa': orig,
        'original_name': amino_acids[orig]['name'],
        'fragment_code': frag_code,
        'fragment_start': start,
        'fragment_end': end,
        'substituted_aa': subs,
        'substituted_name': amino_acids[subs]['name'],
        'modification_description': f"{amino_acids[orig]['name']}->{amino_acids[subs]['name']} on {frag_code} ({start}-{end})"
    }

#print(extract_fragment_id(filtered_avg_116121))
#print(extract_fragment_mod_id(filtered_med_AC7E))

In [11]:
# function to identify the modification fragment
def extract_fragment_modification_id(filename: str) -> str:
    """
    Extract fragment modification identifier from filename.
    Expects a 4 number/letter code where 1: subtituted aa, 2-3: native sequence, 4: aa subtitution.
    Example: 'SS6V.csv' -> 'S modification into V in S6 fragment (116-121)'
    """
    match = re.search(r'(\d{4})', filename)
    if not match:
        raise ValueError(f"Filename {filename} does not contain a 4 letter/number code")
    digits = match.group(1)
    Original_fragment = digits[2:3]
    start, end = digits[:3], digits[3:]
    return f"{start}-{end}"

In [12]:
# function to get the LepR residue position of each identified interaction series of a fragment
def get_position_set_from_row(row: pd.Series) -> Set[int]:
    """
    Get set of residue positions from a single row.
    Uses 'positions' column if available (list), else start/end.
    """
    if 'positions' in row and pd.notna(row['positions']):
        try:
            pos_list = ast.literal_eval(row['positions'])
            return set(pos_list)
        except (ValueError, SyntaxError):
            pass
    if 'start_position' in row and 'end_position' in row:
        start = int(row['start_position'])
        end = int(row['end_position'])
        return set(range(start, end + 1))
    return set()

def get_union_positions_from_df(df: pd.DataFrame) -> Set[int]:
    """Union of all positions across all rows of a DataFrame."""
    all_pos = set()
    for _, row in df.iterrows():
        all_pos.update(get_position_set_from_row(row))
    return all_pos

In [13]:
# function to compare a specific fragment with the original leptin interaction
def filter_fragment_by_native(
    native_df: pd.DataFrame,
    fragment_url: str,
    fragment_id: str = None
) -> pd.DataFrame:
    """
    Filter a fragment CSV to rows that overlap with any position in native_df.
    Adds a 'fragment_id' column.
    
    Parameters:
    - native_df: DataFrame of native leptin (already loaded)
    - fragment_url: URL or path to fragment CSV
    - fragment_id: optional, if not provided it will be extracted from URL
    
    Returns:
    - Filtered DataFrame (may be empty)
    """
    # Get native position set once
    native_positions = get_union_positions_from_df(native_df)
    
    # Load fragment
    frag_df = pd.read_csv(fragment_url)
    if frag_df.empty:
        return pd.DataFrame()
    
    # Determine fragment ID if not given
    if fragment_id is None:
        fragment_id = extract_fragment_id(fragment_url)
    
    # Mask rows that overlap
    keep = []
    for _, row in frag_df.iterrows():
        row_pos = get_position_set_from_row(row)
        overlap = bool(row_pos & native_positions)
        keep.append(overlap)
    
    filtered = frag_df[keep].copy()
    filtered.insert(0, 'fragment_id', fragment_id)  # add column at beginning
    return filtered

In [14]:
# define the native original leptin
native_avg_df = pd.read_csv(filtered_avg_001146)
native_med_df = pd.read_csv(filtered_med_001146)

In [15]:
# define the test fragments of the original hexapeptides
test_fragments_avg = [
    filtered_avg_116121,
    filtered_avg_116130,
    filtered_avg_117122,
    filtered_avg_117125,
    filtered_avg_118123,
    filtered_avg_119124,
    filtered_avg_120125,
    filtered_avg_121126,
    filtered_avg_122127,
    filtered_avg_123128,
    filtered_avg_125130
] #       filtered_avg_124129,

test_fragments_med = [
    filtered_med_116121,
    filtered_med_116130,
    filtered_med_117122,
    filtered_med_117125,
    filtered_med_118123,
    filtered_med_119124,
    filtered_med_120125,
    filtered_med_121126,
    filtered_med_122127,
    filtered_med_123128,
    filtered_med_124129,
    filtered_med_125130
]

In [20]:
# define the test fragments of the original nonapeptides
test_fragments_avg = [
    filtered_avg_116124,
    filtered_avg_117125,
    filtered_avg_118126,
    filtered_avg_119127,
    filtered_avg_120128,
    filtered_avg_121129,
    filtered_avg_122130
] 

test_fragments_med = [
    filtered_med_116124,
    filtered_med_117125,
    filtered_med_118126,
    filtered_med_119127,
    filtered_med_120128,
    filtered_med_121129,
    filtered_med_122130
]

In [16]:
# define the test fragments of SS6 (116-1221) subtitution with natural amino acid
test_fragments_avg = [
    filtered_avg_SS6V,
    filtered_avg_SS6L,
    filtered_avg_SS6I,
    filtered_avg_SS6F,
    filtered_avg_SS6M,
    filtered_avg_SS6W,
    filtered_avg_SS6P,
    filtered_avg_SS6G,
    filtered_avg_SS6C,
    filtered_avg_SS6R,
    filtered_avg_SS6N,
    filtered_avg_SS6D,
    filtered_avg_SS6E,
    filtered_avg_SS6Q,
    filtered_avg_SS6H,
    filtered_avg_SS6K,
    filtered_avg_SS6A,
    filtered_avg_SS6T,
    filtered_avg_SS6Y
]

test_fragments_med = [
    filtered_med_SS6V,
    filtered_med_SS6L,
    filtered_med_SS6I,
    filtered_med_SS6F,
    filtered_med_SS6M,
    filtered_med_SS6W,
    filtered_med_SS6P,
    filtered_med_SS6G,
    filtered_med_SS6C,
    filtered_med_SS6R,
    filtered_med_SS6N,
    filtered_med_SS6D,
    filtered_med_SS6E,
    filtered_med_SS6Q,
    filtered_med_SS6H,
    filtered_med_SS6K,
    filtered_med_SS6A,
    filtered_med_SS6T,
    filtered_med_SS6Y
]

In [17]:
# define the test fragments of AC7 (117-122) subtitution with natural amino acid
test_fragments_avg = [
    filtered_avg_AC7V,
    filtered_avg_AC7L,
    filtered_avg_AC7I,
    filtered_avg_AC7F,
    filtered_avg_AC7M,
    filtered_avg_AC7W,
    filtered_avg_AC7P,
    filtered_avg_AC7G,
    filtered_avg_AC7C,
    filtered_avg_AC7R,
    filtered_avg_AC7N,
    filtered_avg_AC7D,
    filtered_avg_AC7E,
    filtered_avg_AC7Q,
    filtered_avg_AC7H,
    filtered_avg_AC7K,
    filtered_avg_AC7S,
    filtered_avg_AC7T,
    filtered_avg_AC7Y
]

test_fragments_med = [
    filtered_med_AC7V,
    filtered_med_AC7L,
    filtered_med_AC7I,
    filtered_med_AC7F,
    filtered_med_AC7M,
    filtered_med_AC7W,
    filtered_med_AC7P,
    filtered_med_AC7G,
    filtered_med_AC7C,
    filtered_med_AC7R,
    filtered_med_AC7N,
    filtered_med_AC7D,
    filtered_med_AC7E,
    filtered_med_AC7Q,
    filtered_med_AC7H,
    filtered_med_AC7K,
    filtered_med_AC7S,
    filtered_med_AC7T,
    filtered_med_AC7Y
]

In [26]:
# compile all the avg comparison for native fragments
all_filtered_avg = []
for frag in test_fragments_avg:
    filtered = filter_fragment_by_native(native_avg_df, frag)
    if not filtered.empty:
        frag_id = extract_fragment_id(frag)
        # else:
        #    frag_id = "unknown"
        # Add a column with the fragment ID (already done inside filter_fragment_by_native)
        # filter_fragment_by_native already adds 'fragment_id', so we don't need to add again.
        all_filtered_avg.append(filtered)
        print(f"Collected {len(filtered)} rows from {frag_id}")
    else:
        print(f"No overlap for {frag}")

# Combine everything into one DataFrame
if all_filtered_avg:
    combined_avg = pd.concat(all_filtered_avg, ignore_index=False)
    # sort the data based on the average value
    combined_avg = combined_avg.sort_values(by= 'average_value', ascending=False)
    combined_avg = combined_avg.reset_index(drop=True)
    # Save to a single CSV in Drive folder
    save_path = os.path.join(results_folder, 'all_overlaps_avg.csv')
    combined_avg.to_csv(save_path, index=False)
    print(f"\nSaved combined DataFrame with {len(combined_avg)} rows to {save_path}")
else:
    print("No overlapping fragments found.")

Collected 22 rows from 116-124
Collected 24 rows from 117-125
Collected 27 rows from 118-126
Collected 25 rows from 119-127
Collected 25 rows from 120-128
Collected 25 rows from 121-129
Collected 25 rows from 122-130

Saved combined DataFrame with 173 rows to /content/drive/MyDrive/Master/Barcelona/Thesis/Documentation/Fit_comparison_native/all_overlaps_avg.csv


In [24]:
# helper code for modified fragments
def extract_code_from_url(url):
    """Extract the 4-character modification code from a URL."""
    base = url.split('?')[0]
    filename = base.split('/')[-1]
    name_without_ext = filename.rsplit('.', 1)[0]
    return name_without_ext[-4:]  # Assumes the last 4 chars are the code

In [25]:
# compile all the avg comparison for modified fragments
all_filtered_avg = []

for frag in test_fragments_avg:
    code = extract_code_from_url(frag)                     # e.g., 'AC7V'
    mod_info = extract_fragment_mod_id(code)
    fragment_id = f"{mod_info['fragment_start']}-{mod_info['fragment_end']}"
    
    filtered = filter_fragment_by_native(native_med_df, frag, fragment_id=fragment_id)
    
    if not filtered.empty:
        # Add modification details
        for key, val in mod_info.items():
            filtered[key] = val
        # **NEW: store the 4‑letter code in its own column**
        filtered['modification_code'] = code
        all_filtered_avg.append(filtered)
        print(f"Collected {len(filtered)} rows from {code} ({mod_info['modification_description']})")
    else:
        print(f"No overlap for {frag} (code {code})")

# Concatenate, sort, and save
if all_filtered_avg:
    combined_mod_avg = pd.concat(all_filtered_avg, ignore_index=True)
    combined_mod_avg = combined_mod_avg.sort_values(by='average_value', ascending=False)
    combined_mod_avg = combined_mod_avg.reset_index(drop=True)
    save_path = os.path.join(results_folder, 'all_overlaps_ModS6_avg.csv')
    combined_mod_avg.to_csv(save_path, index=False)
    print(f"\nSaved combined DataFrame with {len(combined_mod_avg)} rows to {save_path}")
else:
    print("No overlapping fragments found.")

ValueError: Unknown original amino acid: '6'

In [28]:
# compile all the med comparison
all_filtered_med = []
for frag in test_fragments_med:
    filtered = filter_fragment_by_native(native_med_df, frag)
    if not filtered.empty:
        frag_id = extract_fragment_id(frag)
        # Add a column with the fragment ID (already done inside filter_fragment_by_native, but let's ensure)
        # Actually filter_fragment_by_native already adds 'fragment_id', so we don't need to add again.
        all_filtered_med.append(filtered)
        print(f"Collected {len(filtered)} rows from {frag_id}")
    else:
        print(f"No overlap for {frag}")

# Combine everything into one DataFrame
if all_filtered_med:
    combined_med = pd.concat(all_filtered_med, ignore_index=True)
    # sort the data based on the average value
    combined_med = combined_med.sort_values(by= 'average_value', ascending=False)
    combined_med = combined_med.reset_index(drop=True)
    # Save to a single CSV in Drive folder
    save_path = os.path.join(results_folder, 'all_overlaps_med.csv')
    combined_med.to_csv(save_path, index=False)
    print(f"\nSaved combined DataFrame with {len(combined_med)} rows to {save_path}")
else:
    print("No overlapping fragments found.")

Collected 23 rows from 116-124
Collected 25 rows from 117-125
Collected 28 rows from 118-126
Collected 28 rows from 119-127
Collected 28 rows from 120-128
Collected 27 rows from 121-129
Collected 28 rows from 122-130

Saved combined DataFrame with 187 rows to /content/drive/MyDrive/Master/Barcelona/Thesis/Documentation/Fit_comparison_native/all_overlaps_med.csv


In [ ]:
# compile all the med comparison for modified fragments
all_filtered_med = []

for frag in test_fragments_med:
    code = extract_code_from_url(frag)                     # e.g., 'AC7V'
    mod_info = extract_fragment_mod_id(code)
    fragment_id = f"{mod_info['fragment_start']}-{mod_info['fragment_end']}"
    
    filtered = filter_fragment_by_native(native_med_df, frag, fragment_id=fragment_id)
    
    if not filtered.empty:
        # Add modification details
        for key, val in mod_info.items():
            filtered[key] = val
        # **NEW: store the 4‑letter code in its own column**
        filtered['modification_code'] = code
        # You may also want to keep the range as 'fragment_id' (optional)
        all_filtered_med.append(filtered)
        print(f"Collected {len(filtered)} rows from {code} ({mod_info['modification_description']})")
    else:
        print(f"No overlap for {frag} (code {code})")

# Concatenate, sort, and save
if all_filtered_med:
    combined_mod_med = pd.concat(all_filtered_med, ignore_index=True)
    combined_mod_med = combined_mod_med.sort_values(by='average_value', ascending=False)
    combined_mod_med = combined_mod_med.reset_index(drop=True)
    save_path = os.path.join(results_folder, 'all_overlaps_ModS6_med.csv')
    combined_mod_med.to_csv(save_path, index=False)
    print(f"\nSaved combined DataFrame with {len(combined_mod_med)} rows to {save_path}")
else:
    print("No overlapping fragments found.")

Collected 22 rows from SS6V (Serine->Valine on S6 (116-121))
Collected 25 rows from SS6L (Serine->Leucine on S6 (116-121))
Collected 25 rows from SS6I (Serine->Isoleucine on S6 (116-121))
Collected 26 rows from SS6F (Serine->Phenylalanine on S6 (116-121))
Collected 26 rows from SS6M (Serine->Methionine on S6 (116-121))
Collected 26 rows from SS6W (Serine->Tryptophan on S6 (116-121))
Collected 23 rows from SS6P (Serine->Proline on S6 (116-121))
Collected 20 rows from SS6G (Serine->Glycine on S6 (116-121))
Collected 27 rows from SS6C (Serine->Cysteine on S6 (116-121))
Collected 27 rows from SS6R (Serine->Arginine on S6 (116-121))
Collected 25 rows from SS6N (Serine->Asparagine on S6 (116-121))
Collected 25 rows from SS6D (Serine->Aspartic acid on S6 (116-121))
Collected 25 rows from SS6E (Serine->Glutamic acid on S6 (116-121))
Collected 28 rows from SS6Q (Serine->Glutamine on S6 (116-121))
Collected 27 rows from SS6H (Serine->Histidine on S6 (116-121))
Collected 23 rows from SS6K (Serine

In [29]:
# function to print general outlook

def print_outlook(df, name):
    """
    Print key statistics for a dataframe.
    
    Parameters:
    - df: pandas DataFrame (must have columns 'average_value', 'cumulative_value', 'length')
    - name: string name of the dataframe (e.g., 'combined_avg')
    """
    if df.empty:
        print(f"{name}: DataFrame is empty.")
        return
    
    print(f"\n{'='*50}")
    print(f"General outlook for {name}")
    print(f"{'='*50}")
    print(f"Total rows: {len(df)}")
    print(f"Max average_value: {df['average_value'].max():.4f}")
    print(f"Min average_value: {df['average_value'].min():.4f}")
    print(f"Max cumulative_value: {df['cumulative_value'].max():.4f}")
    print(f"Min cumulative_value: {df['cumulative_value'].min():.4f}")
    print(f"Longest length (residues): {df['length'].max()}")
    print(f"Shortest length (residues): {df['length'].min()}")
    print(f"Average length: {df['length'].mean():.2f}")

def print_top_distinct_fragments(df, name, top_n=3):
    """
    For each unique fragment_id, take the row with the highest average_value,
    then show the top_n fragments by that best average_value.
    """
    if df.empty:
        print(f"{name}: DataFrame is empty.")
        return

    # Group by fragment_id, keep the row with max average_value per fragment
    best_per_fragment = df.loc[df.groupby('fragment_id')['average_value'].idxmax()]
    # Sort by average_value descending
    best_per_fragment = best_per_fragment.sort_values('average_value', ascending=False)
    top_fragments = best_per_fragment.head(top_n)
    
    print(f"\n{'='*60}")
    print(f"Top {top_n} distinct fragments (by highest average_value) in {name}")
    print(f"{'='*60}")
    
    display_cols = ['fragment_id', 'start_position', 'end_position', 'length', 
                    'average_value', 'cumulative_value', 'domain', 'num_residues_in_domain']
    available_cols = [col for col in display_cols if col in df.columns]
    print(top_fragments[available_cols].to_string(index=False))

In [35]:
# print the sumary for avg data
print_outlook(combined_avg, 'combined_avg')
print_top_distinct_fragments(combined_avg, 'combined_avg', top_n=3)


General outlook for combined_avg
Total rows: 173
Max average_value: 0.4109
Min average_value: 0.2774
Max cumulative_value: 10.1211
Min cumulative_value: 1.1514
Longest length (residues): 29
Shortest length (residues): 4
Average length: 7.99

Top 3 distinct fragments (by highest average_value) in combined_avg
fragment_id  start_position  end_position  length  average_value  cumulative_value    domain  num_residues_in_domain
    122-130             859           875      17       0.410926          6.985746 LepR_Box1                       5
    121-129             859           875      17       0.410926          6.985746 LepR_Box1                       5
    118-126             860           876      17       0.381361          6.483142 LepR_Box1                       6


In [36]:
# print the summary for med data
print_outlook(combined_med, 'combined_med')
print_top_distinct_fragments(combined_med, 'combined_med', top_n=3)


General outlook for combined_med
Total rows: 187
Max average_value: 0.4013
Min average_value: 0.2708
Max cumulative_value: 10.1211
Min cumulative_value: 1.1514
Longest length (residues): 29
Shortest length (residues): 4
Average length: 8.96

Top 3 distinct fragments (by highest average_value) in combined_med
fragment_id  start_position  end_position  length  average_value  cumulative_value      domain  num_residues_in_domain
    122-130             859           876      18       0.401322          7.223800   LepR_Box1                       6
    118-126             860           876      17       0.381361          6.483142   LepR_Box1                       6
    121-129             828           848      21       0.374760          7.869962 LepR_FNIII4                       6


In [37]:
# function to aggregrate the data based on the position
def aggregate_by_position_interval(df):
    """
    Group fragments by identical (start_position, end_position).
    Returns a summary DataFrame with one row per interval.
    Handles both modified (with modification_code) and original fragments.
    """
    df = df.sort_values(['start_position', 'end_position']).reset_index(drop=True)
    grouped = df.groupby(['start_position', 'end_position'])
    
    summary_rows = []
    for (start, end), group in grouped:
        domain = group.iloc[0]['domain']
        length = group.iloc[0]['length']
        region_positions = group.iloc[0]['region_positions']
        residues_list = group.iloc[0]['residues']
        pos_res_pairs = group.iloc[0]['pos_res_pairs']
        
        # Collect original_aa and substituted_aa if present
        original_aa = None
        substituted_aa = None
        if 'original_aa' in df.columns:
            unique_orig = group['original_aa'].dropna().unique()
            unique_subs = group['substituted_aa'].dropna().unique()
            original_aa = ';'.join(unique_orig) if len(unique_orig) > 1 else (unique_orig[0] if len(unique_orig) == 1 else None)
            substituted_aa = ';'.join(unique_subs) if len(unique_subs) > 1 else (unique_subs[0] if len(unique_subs) == 1 else None)
        
        # Collect fragment details
        fragments_details = []
        fragment_ids_list = []
        for _, row in group.iterrows():
            # Determine the identifier: use modification_code if available, else fragment_id
            if 'modification_code' in row and pd.notna(row['modification_code']):
                frag_id = row['modification_code']
            else:
                frag_id = row['fragment_id']
            fragment_ids_list.append(frag_id)
            
            frag_detail = {
                'fragment_id': frag_id,          # now uses the code or range
                'cumulative_value': row['cumulative_value'],
                'average_value': row['average_value'],
                'probabilities': row['probabilities']
            }
            if 'modification_description' in row:
                frag_detail['modification_description'] = row['modification_description']
            fragments_details.append(frag_detail)
        
        summary_rows.append({
            'start_position': start,
            'end_position': end,
            'domain': domain,
            'region_positions': region_positions,
            'length': length,
            'residues': residues_list,
            'pos_res_pairs': pos_res_pairs,
            'original_aa': original_aa,
            'substituted_aa': substituted_aa,
            'number_of_fragments': len(group),
            'fragment_ids': fragment_ids_list,          # now a list of codes (if modified)
            'fragment_details': fragments_details
        })
    
    return pd.DataFrame(summary_rows)

# The fragment_details column contains a list of dictionaries. When saving to CSV, it must be converted to a string (e.g., using str() or json.dumps()). For further analysis in Python, keep the list-of-dicts structure in memory.

In [39]:
# overlap the data for native fragments
comb_overlap_avg = aggregate_by_position_interval(combined_avg)
comb_overlap_med = aggregate_by_position_interval(combined_med)
# comb_overlap_avg.head()

# avg data
comb_overlap_avg['fragment_ids'] = comb_overlap_avg['fragment_ids'].apply(lambda x: ';'.join(x))
comb_overlap_avg['fragment_details'] = comb_overlap_avg['fragment_details'].apply(str)  
# Save to a single CSV in Drive folder
save_path = os.path.join(results_folder, 'position_intervals_summary_nona_avg.csv')
comb_overlap_avg.to_csv(save_path, index=False)
print(f"\nSaved combined DataFrame with {len(comb_overlap_avg)} rows to {save_path}")

# med data
comb_overlap_med['fragment_ids'] = comb_overlap_med['fragment_ids'].apply(lambda x: ';'.join(x))
comb_overlap_med['fragment_details'] = comb_overlap_med['fragment_details'].apply(str)  
# Save to a single CSV in Drive folder
save_path = os.path.join(results_folder, 'position_intervals_summary_nona_med.csv')
comb_overlap_med.to_csv(save_path, index=False)
print(f"\nSaved combined DataFrame with {len(comb_overlap_med)} rows to {save_path}")

NameError: name 'aggregate_by_position_interval' is not defined